# CASMI offline retrieval
Attach the competition data and exactly one private CASMI bundle dataset. Use `bundle_np_prior/` for the COCONUT annotation/count variant; `bundle/` retains the earlier baseline. Internet must be disabled. This notebook performs inference only. It chooses the matching config from the bundle and verifies its file hashes. Local NP-domain validation does not establish hidden-test performance.


In [ ]:
import os, sys, subprocess, json, hashlib, time
from pathlib import Path
started = time.monotonic()
# Environment overrides also allow this exact notebook to be smoke-tested locally.
input_root = Path(os.environ.get('CASMI_INPUT_ROOT', '/kaggle/input'))
bundles = list(input_root.rglob('manifest.json')) if 'CASMI_BUNDLE' not in os.environ else [Path(os.environ['CASMI_BUNDLE']) / 'manifest.json']
bundles = [p for p in bundles if (p.parent / 'src/inference.py').exists()]
assert len(bundles) == 1, f'Attach exactly one CASMI bundle; found {bundles}'
bundle = bundles[0].parent
files = list(input_root.rglob('test.parquet')) if 'CASMI_DATA' not in os.environ else [Path(os.environ['CASMI_DATA']) / 'test.parquet']
assert len(files) == 1, f'Expected one competition test file, found {files}'
data = files[0].parent
# Check bundled files before execution, including model/data consistency.
for relative, expected in json.loads((bundle / 'manifest.json').read_text()).items():
    p = bundle / relative
    assert p.stat().st_size == expected['bytes'], relative
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(chunk)
    assert h.hexdigest() == expected['sha256'], relative
try:
    import rdkit
    correct_rdkit = rdkit.__version__ == '2026.03.3'
except ImportError:
    correct_rdkit = False
if not correct_rdkit:
    target = Path('/kaggle/working/casmi_deps')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--find-links', str(bundle / 'wheels'), '--target', str(target), 'rdkit==2026.3.3'], check=True)
    # Use a fresh subprocess below, avoiding a previously imported RDKit version.
    os.environ['PYTHONPATH'] = str(target) + os.pathsep + str(bundle)
else:
    os.environ['PYTHONPATH'] = str(bundle) + os.pathsep + os.environ.get('PYTHONPATH', '')


In [ ]:
output = Path(os.environ.get('CASMI_OUTPUT', '/kaggle/working/submission.csv'))
if (bundle / 'artifacts/candidates_np_prior.parquet').exists():
    config_name = 'np_prior.json'
elif (bundle / 'artifacts/candidates_extended.parquet').exists():
    config_name = 'ensemble.json'
else:
    config_name = 'baseline.json'
config = json.loads((bundle / 'configs' / config_name).read_text())
print('Using config:', config_name)
code = """
import json, sys, rdkit
assert rdkit.__version__ == '2026.03.3', rdkit.__version__
from src.inference import run
run(sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], json.loads(sys.argv[5]))
"""
subprocess.run([sys.executable, '-c', code, str(data / 'test.parquet'), str(data / 'sample_submission.csv'), str(bundle / 'artifacts'), str(output), json.dumps(config)], check=True)
assert output.is_file()
print(output.with_suffix('.stats.json').read_text())
print('End-to-end seconds including integrity check:', time.monotonic() - started)
